In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


REL-01 — Initialize

In [2]:
import os
import json
import math
import pandas as pd
import numpy as np
from datetime import datetime, timezone

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

RELIABILITY_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Reliability"
)

os.makedirs(RELIABILITY_RESULTS_DIR, exist_ok=True)

print("CrisisWeave Reliability Assessment initialized.")
print("Output directory:", RELIABILITY_RESULTS_DIR)

CrisisWeave Reliability Assessment initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Reliability


REL-02 — Load evidence and conflict outputs

In [3]:
# Load unified evidence
evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(evidence_path, "r", encoding="utf-8") as f:
    evidence_record = json.load(f)


# Load conflict assessment
conflict_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Conflict",
    "CrisisWeave_Conflict_Assessment_CW001_EV001.json"
)

with open(conflict_path, "r", encoding="utf-8") as f:
    conflict_assessment = json.load(f)


print("Evidence loaded:", evidence_record["evidence_id"])
print("Conflict assessment loaded:", conflict_assessment["evidence_id"])

Evidence loaded: EV-001
Conflict assessment loaded: EV-001


REL-03 — Define the assessment reference time

In [4]:
latest_timestamp = pd.to_datetime(
    evidence_record["timestamp"]
)

print("Reference assessment time:")
print(latest_timestamp)

Reference assessment time:
2026-09-11 12:00:00


REL-04 — Define evidence timestamps

In [5]:
evidence_timestamps = {
    "CNN": evidence_record["timestamp"],
    "BERT": evidence_record["timestamp"],
    "OpenStreetMap": evidence_record["timestamp"],
    "Temporal_Analysis": evidence_record["timestamp"],
    "Response_Event": evidence_record["response_evidence"][
        "response_timestamp"
    ]
}

evidence_timestamps

{'CNN': '2026-09-11T12:00:00',
 'BERT': '2026-09-11T12:00:00',
 'OpenStreetMap': '2026-09-11T12:00:00',
 'Temporal_Analysis': '2026-09-11T12:00:00',
 'Response_Event': '2026-09-11T12:00:00'}

REL-05 — Freshness calculation

In [6]:
def calculate_freshness(
    evidence_time,
    reference_time,
    decay_rate=0.25
):
    """
    Calculate evidence freshness using exponential decay.

    F = exp(-lambda * age_hours)
    """

    evidence_time = pd.to_datetime(evidence_time)
    reference_time = pd.to_datetime(reference_time)

    age_hours = max(
        0,
        (reference_time - evidence_time).total_seconds() / 3600
    )

    freshness = math.exp(
        -decay_rate * age_hours
    )

    return {
        "age_hours": round(age_hours, 4),
        "freshness_score": round(freshness, 4)
    }


freshness_results = {}

for source, timestamp in evidence_timestamps.items():
    freshness_results[source] = calculate_freshness(
        timestamp,
        latest_timestamp
    )

print(json.dumps(
    freshness_results,
    indent=2
))

{
  "CNN": {
    "age_hours": 0,
    "freshness_score": 1.0
  },
  "BERT": {
    "age_hours": 0,
    "freshness_score": 1.0
  },
  "OpenStreetMap": {
    "age_hours": 0,
    "freshness_score": 1.0
  },
  "Temporal_Analysis": {
    "age_hours": 0,
    "freshness_score": 1.0
  },
  "Response_Event": {
    "age_hours": 0,
    "freshness_score": 1.0
  }
}


REL-06 — Test freshness decay

In [7]:
# Controlled freshness test
# This does NOT modify the real evidence.

test_times = [
    0,
    1,
    2,
    4,
    8,
    24
]

freshness_test = pd.DataFrame([
    {
        "age_hours": hours,
        "freshness_score": round(
            math.exp(-0.25 * hours),
            4
        )
    }
    for hours in test_times
])

print("Freshness decay test:")
display(freshness_test)

Freshness decay test:


,age_hours,freshness_score
0,0,1.0000
1,1,0.7788
2,2,0.6065
3,4,0.3679
4,8,0.1353
5,24,0.0025


REL-07 — Source baseline reliability

In [8]:
source_baseline_reliability = {
    "CNN": 0.75,
    "BERT": 0.75,
    "OpenStreetMap": 0.90,
    "Temporal_Analysis": 0.80,
    "Response_Event": 0.85
}

baseline_table = pd.DataFrame([
    {
        "source": source,
        "baseline_reliability": score
    }
    for source, score
    in source_baseline_reliability.items()
])

display(baseline_table)

,source,baseline_reliability
0,CNN,0.75
1,BERT,0.75
2,OpenStreetMap,0.90
3,Temporal_Analysis,0.80
4,Response_Event,0.85


REL-08 — Combine baseline reliability + freshness

In [9]:
reliability_results = []

for source, baseline in source_baseline_reliability.items():

    freshness = freshness_results[source]["freshness_score"]

    pre_conflict_reliability = (
        baseline * freshness
    )

    reliability_results.append({
        "source": source,
        "baseline_reliability": baseline,
        "age_hours": freshness_results[source]["age_hours"],
        "freshness_score": freshness,
        "pre_conflict_reliability": round(
            pre_conflict_reliability,
            4
        )
    })


reliability_df = pd.DataFrame(
    reliability_results
)

display(reliability_df)

,source,baseline_reliability,age_hours,freshness_score,pre_conflict_reliability
0,CNN,0.75,0,1.0,0.75
1,BERT,0.75,0,1.0,0.75
2,OpenStreetMap,0.90,0,1.0,0.90
3,Temporal_Analysis,0.80,0,1.0,0.80
4,Response_Event,0.85,0,1.0,0.85


REL-09 — Apply conflict penalty

In [10]:
# REL-09 — Apply conflict penalty to source reliability

conflict_status = conflict_assessment[
    "severity_conflict"
]["status"]

conflict_penalties = {
    "AGREEMENT": 0.00,
    "MINOR_CONFLICT": 0.25,
    "MAJOR_CONFLICT": 0.50
}

conflict_penalty = conflict_penalties.get(
    conflict_status,
    0.00
)

reliability_df["conflict_status"] = conflict_status

reliability_df["conflict_penalty"] = conflict_penalty

reliability_df["reliability_after_conflict"] = (
    reliability_df["pre_conflict_reliability"]
    * (1 - reliability_df["conflict_penalty"])
)

reliability_df["reliability_after_conflict"] = (
    reliability_df["reliability_after_conflict"]
    .round(4)
)

display(reliability_df)

,source,baseline_reliability,age_hours,freshness_score,pre_conflict_reliability,conflict_status,conflict_penalty,reliability_after_conflict
0,CNN,0.75,0,1.0,0.75,AGREEMENT,0.0,0.75
1,BERT,0.75,0,1.0,0.75,AGREEMENT,0.0,0.75
2,OpenStreetMap,0.90,0,1.0,0.90,AGREEMENT,0.0,0.90
3,Temporal_Analysis,0.80,0,1.0,0.80,AGREEMENT,0.0,0.80
4,Response_Event,0.85,0,1.0,0.85,AGREEMENT,0.0,0.85


REL-10 — Handle model confidence

In [11]:
# REL-10 — Incorporate model confidence when available

cnn_confidence = evidence_record[
    "image_evidence"
].get("cnn_confidence")

bert_confidence = evidence_record[
    "text_evidence"
].get("bert_confidence")


def confidence_factor(confidence):
    if confidence is None:
        return 1.0

    try:
        confidence = float(confidence)
        return max(0.0, min(1.0, confidence))
    except:
        return 1.0


cnn_conf_factor = confidence_factor(cnn_confidence)
bert_conf_factor = confidence_factor(bert_confidence)

reliability_df["confidence_factor"] = 1.0

reliability_df.loc[
    reliability_df["source"] == "CNN",
    "confidence_factor"
] = cnn_conf_factor

reliability_df.loc[
    reliability_df["source"] == "BERT",
    "confidence_factor"
] = bert_conf_factor

reliability_df["final_reliability"] = (
    reliability_df["reliability_after_conflict"]
    * reliability_df["confidence_factor"]
)

reliability_df["final_reliability"] = (
    reliability_df["final_reliability"]
    .clip(0, 1)
    .round(4)
)

display(reliability_df)

,source,baseline_reliability,age_hours,freshness_score,pre_conflict_reliability,conflict_status,conflict_penalty,reliability_after_conflict,confidence_factor,final_reliability
0,CNN,0.75,0,1.0,0.75,AGREEMENT,0.0,0.75,1.0,0.75
1,BERT,0.75,0,1.0,0.75,AGREEMENT,0.0,0.75,1.0,0.75
2,OpenStreetMap,0.90,0,1.0,0.90,AGREEMENT,0.0,0.90,1.0,0.90
3,Temporal_Analysis,0.80,0,1.0,0.80,AGREEMENT,0.0,0.80,1.0,0.80
4,Response_Event,0.85,0,1.0,0.85,AGREEMENT,0.0,0.85,1.0,0.85


REL-11 — Add temporal consistency

In [12]:
# REL-11 — Temporal consistency assessment

cnn_severity = evidence_record[
    "image_evidence"
]["cnn_severity"]

temporal_severity = evidence_record[
    "temporal_evidence"
].get("latest_severity")

temporal_trend = evidence_record[
    "temporal_evidence"
]["overall_temporal_trend"]


if (
    cnn_severity == temporal_severity
    and cnn_severity is not None
):
    temporal_consistency = 1.0

elif (
    cnn_severity in ["Moderate", "High"]
    and temporal_trend == "INCREASING"
):
    temporal_consistency = 0.8

else:
    temporal_consistency = 0.6


reliability_df["temporal_consistency"] = (
    temporal_consistency
)

display(
    reliability_df[
        [
            "source",
            "final_reliability",
            "temporal_consistency"
        ]
    ]
)

,source,final_reliability,temporal_consistency
0,CNN,0.75,1.0
1,BERT,0.75,1.0
2,OpenStreetMap,0.90,1.0
3,Temporal_Analysis,0.80,1.0
4,Response_Event,0.85,1.0


REL-12 — Calculate final reliability

In [13]:
# REL-12 — Final evidence reliability

reliability_df["final_reliability"] = (
    reliability_df["final_reliability"]
    .clip(0, 1)
    .round(4)
)

print("Final reliability scores:")
display(
    reliability_df[
        [
            "source",
            "baseline_reliability",
            "freshness_score",
            "confidence_factor",
            "conflict_status",
            "conflict_penalty",
            "final_reliability"
        ]
    ]
)

Final reliability scores:


,source,baseline_reliability,freshness_score,confidence_factor,conflict_status,conflict_penalty,final_reliability
0,CNN,0.75,1.0,1.0,AGREEMENT,0.0,0.75
1,BERT,0.75,1.0,1.0,AGREEMENT,0.0,0.75
2,OpenStreetMap,0.90,1.0,1.0,AGREEMENT,0.0,0.90
3,Temporal_Analysis,0.80,1.0,1.0,AGREEMENT,0.0,0.80
4,Response_Event,0.85,1.0,1.0,AGREEMENT,0.0,0.85


REL-13 — Create reliability summary

In [14]:
# REL-13 — Build reliability summary

reliability_summary = {
    "incident_id": evidence_record["incident_id"],
    "evidence_id": evidence_record["evidence_id"],

    "conflict_status": conflict_status,

    "overall_conflict": conflict_assessment[
        "overall_conflict"
    ],

    "sources": reliability_df[
        [
            "source",
            "baseline_reliability",
            "freshness_score",
            "confidence_factor",
            "conflict_penalty",
            "final_reliability"
        ]
    ].to_dict(orient="records")
}

print(json.dumps(
    reliability_summary,
    indent=2,
    ensure_ascii=False
))

{
  "incident_id": "CW-001",
  "evidence_id": "EV-001",
  "conflict_status": "AGREEMENT",
  "overall_conflict": false,
  "sources": [
    {
      "source": "CNN",
      "baseline_reliability": 0.75,
      "freshness_score": 1.0,
      "confidence_factor": 1.0,
      "conflict_penalty": 0.0,
      "final_reliability": 0.75
    },
    {
      "source": "BERT",
      "baseline_reliability": 0.75,
      "freshness_score": 1.0,
      "confidence_factor": 1.0,
      "conflict_penalty": 0.0,
      "final_reliability": 0.75
    },
    {
      "source": "OpenStreetMap",
      "baseline_reliability": 0.9,
      "freshness_score": 1.0,
      "confidence_factor": 1.0,
      "conflict_penalty": 0.0,
      "final_reliability": 0.9
    },
    {
      "source": "Temporal_Analysis",
      "baseline_reliability": 0.8,
      "freshness_score": 1.0,
      "confidence_factor": 1.0,
      "conflict_penalty": 0.0,
      "final_reliability": 0.8
    },
    {
      "source": "Response_Event",
      "baseline_r

REL-14 — Save reliability result

In [15]:
# REL-14 — Save reliability assessment

reliability_json_path = os.path.join(
    RELIABILITY_RESULTS_DIR,
    "CrisisWeave_Reliability_CW001_EV001.json"
)

reliability_csv_path = os.path.join(
    RELIABILITY_RESULTS_DIR,
    "CrisisWeave_Reliability_CW001_EV001.csv"
)

with open(
    reliability_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        reliability_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

reliability_df.to_csv(
    reliability_csv_path,
    index=False
)

print("Reliability JSON saved:")
print(reliability_json_path)

print("\nReliability CSV saved:")
print(reliability_csv_path)

Reliability JSON saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Reliability/CrisisWeave_Reliability_CW001_EV001.json

Reliability CSV saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Reliability/CrisisWeave_Reliability_CW001_EV001.csv
